In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
PROJECT_DIR = Path.cwd().resolve().parent
if not (PROJECT_DIR / 'processed').exists() and (Path.cwd().resolve() / 'Parkinson').exists():
    PROJECT_DIR = Path.cwd().resolve() / 'Parkinson'

EXTRACT_DIR   = PROJECT_DIR / 'processed' / 'extraction'
TRANSFORM_DIR = PROJECT_DIR / 'processed' / 'transform'

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Updated transform outputs
wide_charttime = pd.read_csv(TRANSFORM_DIR / 'events_12h_wide_by_charttime.csv')
binned_timeseries = pd.read_csv(TRANSFORM_DIR / 'events_12h_binned.csv')
events_long = pd.read_csv(TRANSFORM_DIR / 'events_12h_long.csv')
cohort_final = pd.read_csv(TRANSFORM_DIR / 'cohort_final.csv')
cohort_flow = pd.read_csv(TRANSFORM_DIR / 'cohort_attrition.csv')

for df, cols in [
    (wide_charttime, ['charttime', 'intime', 'outtime']),
    (binned_timeseries, ['bin_start', 'bin_end', 'intime', 'outtime']),
    (events_long, ['charttime', 'intime', 'outtime']),
    (cohort_final, ['intime', 'outtime', 'admittime', 'dischtime', 'deathtime']),
]:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')


cohort_stay_ids = set(cohort_final['stay_id'])

print(f"Charttime wide table: {wide_charttime.shape}")
print(f"12h binned table: {binned_timeseries.shape}")
print(f"Long events: {events_long.shape}")
print(f"Cohort stays: {cohort_final['stay_id'].nunique():,}")
display(cohort_flow)


In [ ]:
# Cohort attrition을 subject/stay/window 기준으로 시각화합니다.
cohort_flow_plot = cohort_flow.copy()
cohort_flow_plot['step_label'] = cohort_flow_plot['step'].str.replace(r'^\d+\. ', '', regex=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(cohort_flow_plot['step_label'], cohort_flow_plot['n_subjects'], marker='o', label='Subjects')
axes[0].plot(cohort_flow_plot['step_label'], cohort_flow_plot['n_stays'], marker='o', label='ICU stays')
axes[0].set_title('Cohort attrition')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend()

axes[1].plot(cohort_flow_plot['step_label'], cohort_flow_plot['total_12h_windows'], marker='o', label='Total 12h windows')
candidate_window_col = 'candidate_12h_windows_excl_first48_last12'
axes[1].plot(
    cohort_flow_plot['step_label'],
    cohort_flow_plot[candidate_window_col],
    marker='o',
    label='Candidate windows excl. first48/last12',
)
axes[1].set_title('12-hour windows after each criterion')
axes[1].set_ylabel('Windows')
axes[1].tick_params(axis='x', rotation=25)
axes[1].legend()

for ax in axes:
    ax.margins(x=0.03)

plt.tight_layout()
plt.show()




## EDA: 환자 기본정보

In [ ]:
# 범주형 기본정보는 subject 안에서 가장 흔한 값을 대표값으로 사용합니다.
def first_mode(values):
    mode_values = values.dropna().mode()
    if len(mode_values) == 0:
        return np.nan
    return mode_values.iloc[0]

# subject-level 요약 테이블을 만듭니다.
subject_summary = (
    cohort_final
    .sort_values(['subject_id', 'intime'])
    .groupby('subject_id')
    .agg(
        ever_delirium=('ever_delirium', 'max'),
        n_stays=('stay_id', 'nunique'),
        first_age=('anchor_age', 'first'),
        gender=('gender', 'first'),
        race=('race', first_mode),
        admission_type=('admission_type', first_mode),
        specialty=('specialty', first_mode),
        first_intime=('intime', 'min'),
        last_outtime=('outtime', 'max'),
        total_icu_los_hours=('icu_los_hours', 'sum'),
    )
    .reset_index()
)

stay_summary = cohort_final[['subject_id', 'stay_id', 'anchor_age', 'gender', 'race', 'admission_type', 'icu_los_hours', 'ever_delirium']].copy()

print('=== Cohort overview ===')
print(f"Subjects: {subject_summary['subject_id'].nunique():,}")
print(f"Stays: {cohort_final['stay_id'].nunique():,}")
print(f"ICU stays per subject: median={subject_summary['n_stays'].median():.1f}, IQR={subject_summary['n_stays'].quantile(0.25):.1f}-{subject_summary['n_stays'].quantile(0.75):.1f}")
print(f"Parkinson cohort period: {cohort_final['intime'].min()} to {cohort_final['outtime'].max()}")

print()
print('=== ever_delirium 분포 (subject-level) ===')
print(subject_summary['ever_delirium'].value_counts().sort_index())
print((subject_summary['ever_delirium'].value_counts(normalize=True).sort_index() * 100).round(2))

print()
print('=== Subject-level numeric summary ===')
display(subject_summary[['first_age', 'n_stays', 'total_icu_los_hours']].describe())

print()
print('=== Subject-level summary by ever_delirium ===')
display(subject_summary.groupby('ever_delirium')[['first_age', 'n_stays', 'total_icu_los_hours']].describe())

print()
print('=== Gender by ever_delirium (%) ===')
display((pd.crosstab(subject_summary['ever_delirium'], subject_summary['gender'], normalize='index') * 100).round(1))

print()
print('=== Admission type by ever_delirium (%) ===')
display((pd.crosstab(stay_summary['ever_delirium'], stay_summary['admission_type'], normalize='index') * 100).round(1))

if subject_summary['specialty'].notna().any():
    print()
    print('=== Specialty by ever_delirium (%) ===')
    display((pd.crosstab(subject_summary['ever_delirium'], subject_summary['specialty'], normalize='index') * 100).round(1))


In [ ]:
# 환자 기본정보와 ever_delirium 분포를 시각화합니다.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

subject_summary['ever_delirium'].value_counts().sort_index().rename({0: 'No delirium', 1: 'Ever delirium'}).plot(
    kind='bar', ax=axes[0, 0], color=['#72B7B2', '#E45756']
)
axes[0, 0].set_title('Subject-level ever_delirium')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('Subjects')
axes[0, 0].tick_params(axis='x', rotation=0)

subject_summary.boxplot(column='first_age', by='ever_delirium', ax=axes[0, 1], grid=False)
axes[0, 1].set_title('Age by ever_delirium')
axes[0, 1].set_xlabel('ever_delirium')
axes[0, 1].set_ylabel('Age')

subject_summary.boxplot(column='total_icu_los_hours', by='ever_delirium', ax=axes[1, 0], grid=False)
axes[1, 0].set_title('Total ICU LOS by ever_delirium')
axes[1, 0].set_xlabel('ever_delirium')
axes[1, 0].set_ylabel('Hours')

stay_summary['admission_type'].value_counts().head(8).sort_values().plot(kind='barh', ax=axes[1, 1], color='#59A14F')
axes[1, 1].set_title('Top admission types')
axes[1, 1].set_xlabel('ICU stays')
axes[1, 1].set_ylabel('')

fig.suptitle('')
plt.tight_layout()
plt.show()


## EDA: 섬망 평가 주기

Assessment 횟수, assessment 간격, 첫 assessment까지 걸린 시간, ICU hour/day당 assessment 빈도를 확인합니다.


In [ ]:
# Long-format delirium event row를 assessment 시점으로 사용합니다.
assessments = (
    events_long
    .loc[
        events_long['feature_name'].eq('delirium')
        & events_long['value'].notna()
        & events_long['charttime'].notna(),
        ['subject_id', 'stay_id', 'charttime', 'bin', 'hours', 'value']
    ]
    .rename(columns={'bin': 'assessment_bin', 'hours': 'assessment_hours', 'value': 'delirium'})
    .drop_duplicates(['stay_id', 'charttime', 'delirium'])
    .sort_values(['stay_id', 'charttime'])
    .reset_index(drop=True)
)
assessments['delirium'] = assessments['delirium'].astype('int8')
assessments = assessments.merge(
    cohort_final[['subject_id', 'ever_delirium']].drop_duplicates('subject_id'),
    on='subject_id',
    how='left',
)

# stay/subject별 assessment count와 assessment 간격을 계산합니다.
assessment_counts_by_stay = assessments.groupby('stay_id').size().rename('assessment_count').reset_index()
assessment_counts_by_subject = assessments.groupby('subject_id').size().rename('assessment_count').reset_index()
assessment_intervals = assessments.groupby('stay_id')['charttime'].diff().dt.total_seconds().div(3600).dropna()
first_assessment_hours = assessments.groupby('stay_id')['assessment_hours'].min().rename('first_assessment_hour').reset_index()
assessment_rate = assessment_counts_by_stay.merge(cohort_final[['stay_id', 'icu_los_hours']], on='stay_id', how='left')
assessment_rate['assessments_per_icu_day'] = assessment_rate['assessment_count'] / (assessment_rate['icu_los_hours'] / 24)
assessment_rate['assessments_per_icu_hour'] = assessment_rate['assessment_count'] / assessment_rate['icu_los_hours']

print('=== Delirium assessment rows ===')
print(f"Rows: {len(assessments):,}")
print(f"Stays with assessment: {assessments['stay_id'].nunique():,}")
print(f"Subjects with assessment: {assessments['subject_id'].nunique():,}")
print()
print('Assessment outcome distribution:')
print(assessments['delirium'].value_counts().sort_index())

print()
print('=== Assessment count by stay ===')
display(assessment_counts_by_stay['assessment_count'].describe())

print()
print('=== Assessment count by subject ===')
display(assessment_counts_by_subject['assessment_count'].describe())

print()
print('=== Assessment interval hours within stay ===')
display(assessment_intervals.describe())
print(f"Median/IQR: {assessment_intervals.median():.2f} hours / {assessment_intervals.quantile(0.25):.2f}-{assessment_intervals.quantile(0.75):.2f}")

print()
print('=== First assessment hour within ICU stay ===')
display(first_assessment_hours['first_assessment_hour'].describe())

print()
print('=== Assessment frequency ===')
display(assessment_rate[['assessments_per_icu_hour', 'assessments_per_icu_day']].describe())


In [ ]:
# Assessment 빈도와 timing을 시각화합니다.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

assessment_counts_by_stay['assessment_count'].plot(kind='hist', bins=30, ax=axes[0, 0], color='#4C78A8')
axes[0, 0].set_title('Assessment count per ICU stay')
axes[0, 0].set_xlabel('Assessments')
axes[0, 0].set_ylabel('ICU stays')

first_assessment_hours['first_assessment_hour'].plot(kind='hist', bins=30, ax=axes[0, 1], color='#F58518')
axes[0, 1].axvline(24, color='black', linestyle='--', linewidth=1)
axes[0, 1].set_title('First assessment hour')
axes[0, 1].set_xlabel('Hours from ICU admission')
axes[0, 1].set_ylabel('ICU stays')

assessment_intervals.clip(upper=72).plot(kind='hist', bins=30, ax=axes[1, 0], color='#54A24B')
axes[1, 0].set_title('Assessment interval within stay')
axes[1, 0].set_xlabel('Hours, clipped at 72')
axes[1, 0].set_ylabel('Intervals')

assessments['delirium'].value_counts().sort_index().rename({0: 'Negative', 1: 'Positive'}).plot(
    kind='bar', ax=axes[1, 1], color=['#72B7B2', '#E45756']
)
axes[1, 1].set_title('Assessment-level delirium labels')
axes[1, 1].set_xlabel('')
axes[1, 1].set_ylabel('Assessments')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


## EDA: 검사실(lab) 측정 주기

Lab feature별 측정 횟수, 측정 간격, stay/subject coverage와 전체 lab 측정 간격을 확인합니다.


In [ ]:
# cohort-filtered long-format 이벤트에서 lab row만 골라 측정 주기를 계산합니다.
lab_events = events_long[
    events_long['source_table'].eq('labevents')
    & events_long['stay_id'].isin(cohort_stay_ids)
    & events_long['charttime'].notna()
].copy()

lab_events = lab_events.merge(
    cohort_final[['stay_id', 'subject_id', 'intime', 'outtime']],
    on=['stay_id', 'subject_id'],
    how='inner',
    suffixes=('', '_cohort'),
)
lab_events['lab_hour'] = (lab_events['charttime'] - lab_events['intime']).dt.total_seconds() / 3600
lab_events = lab_events[(lab_events['lab_hour'] >= 0) & (lab_events['charttime'] <= lab_events['outtime'])].copy()

# feature별 측정 횟수와 coverage를 요약합니다.
lab_feature_counts = (
    lab_events
    .groupby('feature_name')
    .agg(
        n_measurements=('feature_name', 'size'),
        n_stays=('stay_id', 'nunique'),
        n_subjects=('subject_id', 'nunique'),
    )
    .reset_index()
)
lab_feature_counts['stay_coverage_pct'] = lab_feature_counts['n_stays'] / cohort_final['stay_id'].nunique() * 100
lab_feature_counts['subject_coverage_pct'] = lab_feature_counts['n_subjects'] / cohort_final['subject_id'].nunique() * 100

# 같은 stay 안에서 같은 lab feature가 반복 측정된 간격을 계산합니다.
lab_sorted = lab_events.sort_values(['stay_id', 'feature_name', 'charttime'])
lab_sorted['interval_hours'] = lab_sorted.groupby(['stay_id', 'feature_name'])['charttime'].diff().dt.total_seconds() / 3600
lab_interval_summary = (
    lab_sorted
    .dropna(subset=['interval_hours'])
    .groupby('feature_name')['interval_hours']
    .agg(
        interval_n='count',
        interval_median_hours='median',
        interval_q1_hours=lambda x: x.quantile(0.25),
        interval_q3_hours=lambda x: x.quantile(0.75),
    )
    .reset_index()
)

lab_summary = (
    lab_feature_counts
    .merge(lab_interval_summary, on='feature_name', how='left')
    .sort_values(['n_measurements', 'stay_coverage_pct'], ascending=False)
    .reset_index(drop=True)
)

# 어떤 lab이든 측정된 시점 기준의 전체 lab 간격도 함께 봅니다.
any_lab_times = lab_events[['stay_id', 'charttime']].drop_duplicates().sort_values(['stay_id', 'charttime']).copy()
any_lab_intervals = any_lab_times.groupby('stay_id')['charttime'].diff().dt.total_seconds().div(3600).dropna()

print('=== Lab measurement overview ===')
print(f"Lab rows: {len(lab_events):,}")
print(f"Lab features: {lab_events['feature_name'].nunique():,}")
print(f"Stays with any lab: {lab_events['stay_id'].nunique():,} / {cohort_final['stay_id'].nunique():,}")
print(f"Subjects with any lab: {lab_events['subject_id'].nunique():,} / {cohort_final['subject_id'].nunique():,}")

print()
print('=== Lab feature measurement summary ===')
display(lab_summary)

print()
print('=== Any-lab interval hours within stay ===')
display(any_lab_intervals.describe())
print(f"Median/IQR: {any_lab_intervals.median():.2f} hours / {any_lab_intervals.quantile(0.25):.2f}-{any_lab_intervals.quantile(0.75):.2f}")


In [ ]:
# Lab coverage와 반복 측정 간격을 시각화합니다.
top_lab_coverage = lab_summary.sort_values('stay_coverage_pct', ascending=False).head(20).sort_values('stay_coverage_pct')
top_lab_interval = (
    lab_summary
    .dropna(subset=['interval_median_hours'])
    .sort_values('interval_median_hours')
    .head(20)
    .sort_values('interval_median_hours')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_lab_coverage.plot(x='feature_name', y='stay_coverage_pct', kind='barh', ax=axes[0], legend=False, color='#4C78A8')
axes[0].set_title('Top lab stay coverage')
axes[0].set_xlabel('Stay coverage (%)')
axes[0].set_ylabel('')

if len(top_lab_interval) > 0:
    top_lab_interval.plot(x='feature_name', y='interval_median_hours', kind='barh', ax=axes[1], legend=False, color='#F58518')
    axes[1].set_title('Shortest median lab intervals')
    axes[1].set_xlabel('Median interval hours')
    axes[1].set_ylabel('')
else:
    axes[1].text(0.5, 0.5, 'No repeated lab intervals', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()


## EDA: Feature missingness

Hourly timeseries 기준 feature별 관측률을 확인합니다. Static identifier/time/outcome 컬럼은 제외하고 봅니다.


In [ ]:
# 12시간 bin-level wide table의 feature missingness를 요약하고 시각화합니다.
non_feature_cols = {
    'subject_id', 'hadm_id', 'stay_id', 'bin', 'hours', 'bin_start', 'bin_end',
    'intime', 'outtime', 'admittime', 'dischtime', 'deathtime', 'ever_delirium',
    'split', 'race', 'admission_type', 'gender', 'age', 'anchor_age', 'icu_los_hours',
    'los_hours', 'hospital_expire_flag', 'delirium'
}
feature_cols = [col for col in binned_timeseries.columns if col not in non_feature_cols]

feature_missing = pd.DataFrame({
    'feature_name': feature_cols,
    'observed_rows': binned_timeseries[feature_cols].notna().sum().values,
})
feature_missing['missing_rows'] = len(binned_timeseries) - feature_missing['observed_rows']
feature_missing['observed_pct'] = feature_missing['observed_rows'] / len(binned_timeseries) * 100
feature_missing['missing_pct'] = 100 - feature_missing['observed_pct']
feature_missing = feature_missing.sort_values('observed_pct', ascending=False).reset_index(drop=True)

display(feature_missing)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
feature_missing.head(20).sort_values('observed_pct').plot(
    x='feature_name', y='observed_pct', kind='barh', ax=axes[0], legend=False, color='#4C78A8'
)
axes[0].set_title('Most observed 12h-bin features')
axes[0].set_xlabel('Observed rows (%)')
axes[0].set_ylabel('')

feature_missing.tail(20).sort_values('observed_pct').plot(
    x='feature_name', y='observed_pct', kind='barh', ax=axes[1], legend=False, color='#E45756'
)
axes[1].set_title('Least observed 12h-bin features')
axes[1].set_xlabel('Observed rows (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()
